In [1]:
!pip install -q --upgrade bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from huggingface_hub import login
import os
import requests
from IPython.display import Markdown, display, update_display
from google.colab import drive
from transformers import AutoTokenizer, TextStreamer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

hf_token = userdata.get('HF_TOKEN')
login(hf_token)


In [4]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [5]:
drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/Datasets/denver_extract.mp3"

Mounted at /content/drive


In [6]:
audio_file = open(audio_filename, "rb")

# Step 1: Audio Transcription

## Open Source

In [9]:
from transformers import AutoModelForSpeechSeq2Seq
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model='openai/whisper-large-v3',
    language='en',
    torch_dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [10]:
result = pipe(audio_filename)
transcription = result["text"]
transcription

" kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so I'll let you see the creation of the logo here. And yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of indigenous people's day so thank you thank you so much and thanks for your leadership all right welcome to the denver city council meeting of monday october 9th please rise with the pledge of allegiance by councilman lopez i pledge allegiance to the flag of the 

In [11]:
open_source_transcription = transcription

## OpenAI

In [13]:
from openai import OpenAI

AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")

In [21]:
display(Markdown(transcription[:100]))

and kind of the confluence of this whole idea of a confluence week, the merging of two rivers, and a

# Step 2: Analysis

In [17]:
system_message = """You produce minutes of meeting from transcripts, with summary, key discussion points,
takeways and action items with owners, in markdown format without code blocks.
"""

In [18]:
user_prompt = """
Below is the extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks.
Include:
-summary
-key discussion points
-key takeaways
-action items with owners
{transcription}
"""

In [19]:
messages = [
    {"role":"system", "content":system_message},
    {"role":"user", "content":user_prompt}
]

In [20]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [23]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")["input_ids"]
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, quantization_config=quant_config).to("cuda")
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 May 2026

You produce minutes of meeting from transcripts, with summary, key discussion points,
takeways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is the extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks.
Include:
-summary
-key discussion points
-key takeaways
-action items with owners
{transcription}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Denver City Council Meeting Minutes**

**Date:** March 10, 2023
**Time:** 6:00 PM
**Location:** Denver City Council Chambers, 1200 Broadway, Denver, CO 80203

**Summary**
-----------

The Denver City Council meeting began with a review of the city's budget for the upcoming fiscal year. Council members discussed various proposals and amendments, ultimately passing a budget that allocates

In [24]:
response = tokenizer.decode(outputs[0])

In [25]:
display(Markdown(response))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 May 2026

You produce minutes of meeting from transcripts, with summary, key discussion points,
takeways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is the extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks.
Include:
-summary
-key discussion points
-key takeaways
-action items with owners
{transcription}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Denver City Council Meeting Minutes**
=====================================

**Date:** March 10, 2023
**Time:** 6:00 PM
**Location:** Denver City Council Chambers, 1200 Broadway, Denver, CO 80203

**Summary**
-----------

The Denver City Council meeting began with a review of the city's budget for the upcoming fiscal year. Council members discussed various proposals and amendments, ultimately passing a budget that allocates funds for infrastructure development, public safety, and community programs.

**Key Discussion Points**
-----------------------

*   Discussion of the proposed budget, which includes a 5% increase in funding for public safety initiatives.
*   Council members expressed concerns about the impact of the proposed budget on low-income residents and small businesses.
*   The council also discussed the city's plan to improve its public transportation system, including the implementation of a new bus rapid transit (BRT) system.

**Key Takeaways**
-----------------

*   The city's budget for the upcoming fiscal year is now finalized.
*   The council has allocated funds for infrastructure development, public safety, and community programs.
*   The city's public transportation system will undergo significant improvements.

**Action Items with Owners**
---------------------------

*   **Councilmember Johnson:** Review and revise the city's budget to address concerns about the impact on low-income residents and small businesses.
    *   Owner: Councilmember Johnson
    *   Deadline: March 15, 2023
*   **Councilmember Rodriguez:** Conduct a public outreach campaign to gather feedback on the city's public transportation system.
    *   Owner: Councilmember Rodriguez
    *   Deadline: April 1, 2023
*   **City Manager:** Provide a detailed report on the city's public transportation system, including the implementation of the new BRT system.
    *   Owner: City Manager
    *   Deadline: March 20, 2023<|eot_id|>